# 10 — Matplotlib & Seaborn: Seeing Your Data

| | |
|---|---|
| **Difficulty** | ⭐⭐ (Beginner–Intermediate) |
| **Estimated Time** | 3 hours |
| **Week** | Week 3 — Statistics for ML & Python Data Stack |
| **Prerequisites** | Python basics, NumPy, Pandas (Notebook 09) |

---

## What You Will Learn
By the end of this notebook you will be able to:
- Explain the difference between matplotlib and seaborn and choose the right tool
- Create publication-quality plots using the `fig, ax = plt.subplots()` pattern
- Choose the correct chart type for your data and question
- Build a complete 6-panel EDA dashboard for a house price dataset
- Annotate charts to highlight key data points

## 1. Why Does This Matter for Machine Learning?

> **"A picture is worth 1 000 statistics."**

Numbers alone can deceive you. The famous **Anscombe's Quartet** (1973) showed four datasets with *identical* mean, variance, and correlation — yet they look completely different when plotted. Without visualisation, you might fit a linear model to a dataset that is clearly non-linear.

Visualisation is essential at **three stages** of the ML workflow:

| Stage | What you visualise | Why |
|-------|--------------------|-----|
| **EDA** (Exploratory Data Analysis) | Distributions, outliers, correlations | Understand your data before modelling |
| **Feature engineering** | Feature relationships, skewness | Decide which transforms to apply |
| **Model evaluation** | Residuals, learning curves, confusion matrices | Diagnose what your model is getting wrong |

This notebook focuses on **EDA visualisation** — the foundation for all later ML work.

## 2. Real-World Analogy — Would You Buy a House Sight Unseen?

Imagine you are house hunting. An agent sends you a spec sheet:

```
Bedrooms: 3 | Bathrooms: 2 | Size: 1 850 sqft | Price: $320 000
```

Would you make an offer based only on that? **Of course not.** You want photos, a floor plan, a street view, a neighbourhood map — *visual representations* of the data.

Data science works the same way. The raw numbers are the spec sheet. **Charts are the photos.** They reveal:
- Are there any houses with obviously wrong prices (outliers)?
- Is price roughly normally distributed, or skewed toward expensive homes?
- Does a larger house consistently mean a higher price, or is the relationship noisy?
- Which neighbourhood has the widest price variation?

None of these questions are easy to answer by scanning a spreadsheet — but every one of them jumps out of a well-chosen chart.

## 3. Matplotlib vs Seaborn — Which Tool When?

Think of it like building with LEGOs:

| | Matplotlib | Seaborn |
|---|---|---|
| **Metaphor** | Individual LEGO bricks | Pre-built LEGO sets |
| **Control** | Total — every pixel | Sensible defaults, limited low-level tweaks |
| **Code verbosity** | Verbose (5–15 lines for a nice plot) | Concise (1–3 lines for a nice plot) |
| **Statistical plots** | Manual | Built-in (histplot with KDE, boxplot, violin, etc.) |
| **Best for** | Custom, non-standard plots; fine-grained control | Statistical EDA; quick, beautiful plots |
| **Underlying engine** | Itself | Wraps matplotlib |

**Rule of thumb:** Start with seaborn. Drop down to matplotlib when you need a specific customisation that seaborn does not expose.

### The `fig, ax` pattern — always use it
```python
# PREFERRED: explicit figure and axes objects
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x, y)
ax.set_title('My Plot')
plt.show()

# AVOID: implicit pyplot interface (confusing for multi-panel figures)
plt.plot(x, y)    # which axes does this go to?
plt.title('...')
plt.show()
```
The explicit `fig, ax` pattern is **mandatory** in professional code and any multi-panel figure.

## 4. Setup — Imports and Synthetic Dataset

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker   # for custom axis tick formatting
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# ── Global seaborn theme ──────────────────────────────────────────────────────
# Set ONCE at the top of your notebook; affects all seaborn plots automatically
sns.set_theme(
    style='whitegrid',    # white background with light grid lines
    palette='husl',       # high-contrast, colourblind-friendly colour cycle
    font_scale=1.1        # slightly larger default font
)

# Matplotlib default figure size (avoid tiny plots in Jupyter)
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize']  = 13
plt.rcParams['axes.labelsize']  = 11

# Fix random seed for reproducibility
np.random.seed(42)

print('Matplotlib version:', plt.matplotlib.__version__)
print('Seaborn version:   ', sns.__version__)

In [ ]:
# ── Synthetic house price dataset (200 rows) ──────────────────────────────────
# Same generation logic as Notebook 09 so you can cross-reference
N = 200

neighborhoods = np.random.choice(
    ['Greenwood', 'Downtown', 'Riverside', 'Suburbia'],
    size=N, p=[0.30, 0.20, 0.25, 0.25]
)
sqft      = np.random.normal(1800, 500, N).clip(600, 4500).astype(int)
bedrooms  = np.random.choice([1, 2, 3, 4, 5, 6], size=N,
                              p=[0.05, 0.20, 0.40, 0.25, 0.07, 0.03])
bathrooms = (bedrooms * np.random.uniform(0.5, 0.9, N)).clip(1, 5).astype(int)
age_years = np.random.randint(0, 60, N)
garage    = np.random.choice([0, 1], N, p=[0.25, 0.75])

# Neighbourhood price premiums
premium_map = {'Greenwood': 50_000, 'Downtown': 80_000,
               'Riverside': 30_000, 'Suburbia': 10_000}
premiums = np.array([premium_map[n] for n in neighborhoods])

price = (
    100 * sqft + 15_000 * bedrooms - 1_000 * age_years
    + 20_000 * garage + premiums
    + np.random.normal(0, 20_000, N)
).clip(80_000, 1_200_000).astype(int)

# Monthly time series: assign each house a random sale month in 2023
month_num = np.random.randint(1, 13, N)   # 1 = Jan, 12 = Dec

df = pd.DataFrame({
    'house_id':     [f'H{i:03d}' for i in range(1, N + 1)],
    'neighborhood': neighborhoods,
    'sqft':         sqft,
    'bedrooms':     bedrooms,
    'bathrooms':    bathrooms,
    'age_years':    age_years,
    'garage':       garage,
    'price':        price,
    'month':        month_num,
})

# Derived columns used in several charts
df['price_per_sqft'] = (df['price'] / df['sqft']).round(0).astype(int)
df['log_price']      = np.log(df['price'])

# Monthly average price (for the time-series line plot)
monthly_avg = df.groupby('month')['price'].mean().reset_index()
monthly_avg.columns = ['month', 'avg_price']

print('Dataset shape:', df.shape)
df.head(3)

## Part 1 — Matplotlib Fundamentals

We will build five chart types that every data scientist needs to know. Each uses the `fig, ax = plt.subplots()` pattern.

### 1.1 Line Plot — House Prices Over Time

**When to use:** Continuous variable on the x-axis (usually time); you want to show a trend.

In [ ]:
# ── Line plot: average house price by month ───────────────────────────────────
# Step 1: Always create the figure and axes objects explicitly
fig, ax = plt.subplots(figsize=(11, 5))

# Step 2: Draw the line — marker='o' adds a dot at each data point
ax.plot(
    monthly_avg['month'],     # x: month numbers 1–12
    monthly_avg['avg_price'], # y: average price in that month
    marker='o',               # circle marker at each data point
    linewidth=2.5,            # thickness of the line
    markersize=8,
    color='steelblue',
    label='Monthly Average'
)

# Step 3: Add a horizontal reference line at the annual mean
annual_mean = df['price'].mean()
ax.axhline(annual_mean, color='tomato', linestyle='--', linewidth=1.5, label=f'Annual Mean (${annual_mean:,.0f})')

# Step 4: Annotate the highest-price month
peak_month = monthly_avg.loc[monthly_avg['avg_price'].idxmax()]
ax.annotate(
    f"Peak: ${peak_month['avg_price']:,.0f}",
    xy=(peak_month['month'], peak_month['avg_price']),
    xytext=(peak_month['month'] + 0.5, peak_month['avg_price'] + 8_000),
    arrowprops=dict(arrowstyle='->', color='black'),
    fontsize=10
)

# Step 5: Labels and formatting
ax.set_title('Average House Sale Price by Month — 2023', fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Average Sale Price ($)')
ax.set_xticks(range(1, 13))   # force integer ticks 1–12
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                    'Jul','Aug','Sep','Oct','Nov','Dec'])
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.legend()
ax.grid(True, alpha=0.3)   # light grid for readability

plt.tight_layout()
plt.show()

### 1.2 Scatter Plot — Square Footage vs Price

**When to use:** Relationship between two continuous variables. Essential for spotting linear relationships (important for linear regression), outliers, and clusters.

In [ ]:
# ── Scatter plot: sqft vs price, coloured by neighbourhood ────────────────────
fig, ax = plt.subplots(figsize=(11, 6))

# Use a colour map so each neighbourhood gets a distinct colour
hoods  = df['neighborhood'].unique()
colors = sns.color_palette('husl', len(hoods))

for hood, color in zip(hoods, colors):
    subset = df[df['neighborhood'] == hood]
    ax.scatter(
        subset['sqft'],
        subset['price'],
        c=[color],          # colour for this neighbourhood
        label=hood,
        alpha=0.65,         # slight transparency so overlapping points are visible
        edgecolors='white', # white border makes points pop
        linewidths=0.4,
        s=60                # marker size in points^2
    )

# ── Annotate the single most expensive house ─────────────────────────────────
# In a real EDA report you would highlight outliers and investigate them
most_expensive = df.loc[df['price'].idxmax()]
ax.annotate(
    f"Most expensive\n{most_expensive['house_id']}\n${most_expensive['price']:,.0f}",
    xy=(most_expensive['sqft'], most_expensive['price']),
    xytext=(most_expensive['sqft'] - 600, most_expensive['price'] - 60_000),
    arrowprops=dict(arrowstyle='->', color='black', lw=1.5),
    fontsize=9,
    bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='black', alpha=0.8)
)

ax.set_title('House Size vs Sale Price by Neighbourhood', fontweight='bold')
ax.set_xlabel('House Size (sq ft)')
ax.set_ylabel('Sale Price ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend(title='Neighbourhood', loc='upper left')

plt.tight_layout()
plt.show()

# What to look for in this chart:
# 1. Positive slope = larger houses cost more (expected)
# 2. Downtown points cluster higher for same sqft (location premium)
# 3. Any points far from the trend = potential data errors or special properties

### 1.3 Histogram — Price Distribution

**When to use:** Distribution of a single continuous variable. Before fitting any ML model, you need to understand whether your features/target are normally distributed, skewed, bimodal, etc.

In [ ]:
# ── Histogram: raw price vs log-transformed price ─────────────────────────────
# Showing both side-by-side demonstrates WHY log-transform is useful

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: raw price (likely right-skewed)
axes[0].hist(
    df['price'],
    bins=30,           # number of bars; too few = hide detail, too many = noise
    color='steelblue',
    edgecolor='white',
    linewidth=0.5,
    alpha=0.85
)
# Add vertical lines for mean and median — if they differ a lot, data is skewed
axes[0].axvline(df['price'].mean(),   color='tomato',  linestyle='--', lw=2, label=f"Mean ${df['price'].mean():,.0f}")
axes[0].axvline(df['price'].median(), color='orange',  linestyle='-',  lw=2, label=f"Median ${df['price'].median():,.0f}")
axes[0].set_title('Raw Price Distribution', fontweight='bold')
axes[0].set_xlabel('Sale Price ($)')
axes[0].set_ylabel('Frequency (number of houses)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
axes[0].legend()

# Right: log-transformed price (should be more symmetric)
axes[1].hist(
    df['log_price'],
    bins=30,
    color='mediumseagreen',
    edgecolor='white',
    linewidth=0.5,
    alpha=0.85
)
axes[1].axvline(df['log_price'].mean(),   color='tomato', linestyle='--', lw=2, label=f"Mean {df['log_price'].mean():.2f}")
axes[1].axvline(df['log_price'].median(), color='orange', linestyle='-',  lw=2, label=f"Median {df['log_price'].median():.2f}")
axes[1].set_title('Log-Transformed Price Distribution', fontweight='bold')
axes[1].set_xlabel('log(Sale Price)')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.suptitle('Effect of Log Transform on Right-Skewed Price Data',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Price skewness (raw):  {df["price"].skew():.3f}   (>0 = right-skewed)')
print(f'Price skewness (log):  {df["log_price"].skew():.3f}  (closer to 0 = more symmetric)')

### 1.4 Bar Chart — Average Price per Neighbourhood

**When to use:** Comparing a numeric quantity across distinct categories.

In [ ]:
# ── Horizontal bar chart: average price per neighbourhood ─────────────────────
# Horizontal bars are easier to read when category names are long

avg_by_hood = (
    df.groupby('neighborhood')['price']
      .mean()
      .sort_values(ascending=True)   # ascending so the tallest bar is at the top
)

fig, ax = plt.subplots(figsize=(10, 5))

# barh = horizontal bar chart
bars = ax.barh(
    avg_by_hood.index,    # y-axis: neighbourhood names
    avg_by_hood.values,   # x-axis: average price
    color=sns.color_palette('husl', len(avg_by_hood)),
    edgecolor='black',
    linewidth=0.5,
    height=0.55
)

# Add value labels at the end of each bar for precise reading
for bar in bars:
    width = bar.get_width()
    ax.text(
        width + 3_000,                   # x position: slightly right of bar end
        bar.get_y() + bar.get_height() / 2,   # y position: centre of bar
        f'${width:,.0f}',
        va='center', fontsize=10
    )

ax.set_title('Average Sale Price by Neighbourhood', fontweight='bold')
ax.set_xlabel('Average Sale Price ($)')
ax.set_ylabel('')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.set_xlim(0, avg_by_hood.max() * 1.15)   # leave room for value labels

plt.tight_layout()
plt.show()

### 1.5 Multi-Panel Figure with `plt.subplots`

Real EDA reports show many charts together. `plt.subplots(nrows, ncols)` creates a grid of axes.

In [ ]:
# ── 2 × 2 subplot grid — four charts at once ─────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ── Plot 1 (top-left): Price histogram ───────────────────────────────────────
axes[0, 0].hist(df['price'], bins=25, color='steelblue', edgecolor='white', alpha=0.85)
axes[0, 0].set_title('Price Distribution')
axes[0, 0].set_xlabel('Price ($)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

# ── Plot 2 (top-right): Bedrooms frequency count ─────────────────────────────
bed_counts = df['bedrooms'].value_counts().sort_index()
axes[0, 1].bar(bed_counts.index, bed_counts.values,
               color=sns.color_palette('husl', len(bed_counts)),
               edgecolor='black', linewidth=0.4)
axes[0, 1].set_title('Number of Bedrooms')
axes[0, 1].set_xlabel('Bedrooms')
axes[0, 1].set_ylabel('Count')

# ── Plot 3 (bottom-left): Age vs Price scatter ────────────────────────────────
axes[1, 0].scatter(df['age_years'], df['price'],
                   alpha=0.4, color='mediumseagreen', edgecolors='white', linewidths=0.3, s=40)
axes[1, 0].set_title('House Age vs Sale Price')
axes[1, 0].set_xlabel('Age (years)')
axes[1, 0].set_ylabel('Price ($)')
axes[1, 0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

# ── Plot 4 (bottom-right): Garage presence — price comparison ─────────────────
# Box plot without seaborn — manual grouping
no_garage  = df[df['garage'] == 0]['price']
has_garage = df[df['garage'] == 1]['price']
bp = axes[1, 1].boxplot([no_garage, has_garage],
                         labels=['No Garage', 'Has Garage'],
                         patch_artist=True)
bp['boxes'][0].set_facecolor('#e74c3c')
bp['boxes'][1].set_facecolor('#2ecc71')
for box in bp['boxes']:
    box.set_alpha(0.7)
axes[1, 1].set_title('Price: Garage vs No Garage')
axes[1, 1].set_ylabel('Price ($)')
axes[1, 1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

# Overall title for the figure
fig.suptitle('House Price Dataset — Matplotlib EDA Dashboard', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Part 2 — Seaborn for Statistical Visualisation

Seaborn builds on matplotlib to provide **statistical graphics** in fewer lines of code. Each seaborn function accepts a DataFrame (`data=`) and column names (`x=`, `y=`, `hue=`), which makes exploration very fast.

### 2.1 `sns.histplot` — Histogram with KDE Overlay

A **Kernel Density Estimate (KDE)** is a smoothed version of the histogram. It shows the underlying continuous probability distribution, making it easier to see whether data is bell-shaped, skewed, or bimodal.

In [ ]:
# ── sns.histplot with KDE: one line for a professional-looking distribution plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: price distribution coloured by neighbourhood
sns.histplot(
    data=df,
    x='price',
    hue='neighborhood',      # colour-code by neighbourhood
    kde=True,                # overlay a smooth KDE curve
    bins=20,
    alpha=0.5,               # transparency so overlapping bars are visible
    ax=axes[0]
)
axes[0].set_title('Price Distribution by Neighbourhood', fontweight='bold')
axes[0].set_xlabel('Sale Price ($)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

# Right: single distribution with KDE for price_per_sqft
sns.histplot(
    data=df,
    x='price_per_sqft',
    kde=True,
    color='steelblue',
    bins=25,
    ax=axes[1]
)
axes[1].axvline(df['price_per_sqft'].mean(), color='tomato', linestyle='--', lw=2,
                label=f"Mean ${df['price_per_sqft'].mean():.0f}/sqft")
axes[1].set_title('Price per Sq Ft Distribution', fontweight='bold')
axes[1].set_xlabel('Price per Sq Ft ($)')
axes[1].legend()

plt.suptitle('seaborn histplot — Histogram + KDE', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 2.2 `sns.boxplot` — Price Distribution by Number of Bedrooms

A **box plot** summarises a distribution in 5 numbers:
```
        ○  ← outlier (beyond 1.5 × IQR)
    ┌───┬───┐
    │   │   │  ← box = 25th to 75th percentile (IQR)
 ───┤   │   ├───  ← whiskers
    │   │   │
    └───┴───┘
        ▲
      median (50th percentile)
```

Perfect for comparing distributions across categories.

In [ ]:
# ── sns.boxplot: price by number of bedrooms ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: boxplot — compact, shows median and quartiles precisely
sns.boxplot(
    data=df,
    x='bedrooms',         # categorical x-axis
    y='price',            # continuous y-axis
    palette='husl',       # one colour per bedroom count
    width=0.5,
    ax=axes[0]
)
axes[0].set_title('Price Distribution by Bedrooms\n(Box Plot)', fontweight='bold')
axes[0].set_xlabel('Number of Bedrooms')
axes[0].set_ylabel('Sale Price ($)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

# Right: add individual data points on top (strip plot) to show raw data
# This hybrid is called a "strip + box" plot
sns.boxplot(
    data=df, x='bedrooms', y='price',
    palette='husl', width=0.5, ax=axes[1]
)
sns.stripplot(
    data=df, x='bedrooms', y='price',
    color='black', alpha=0.25, size=3, jitter=True,
    ax=axes[1]
)
axes[1].set_title('Price Distribution by Bedrooms\n(Box + Strip Plot)', fontweight='bold')
axes[1].set_xlabel('Number of Bedrooms')
axes[1].set_ylabel('Sale Price ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

plt.tight_layout()
plt.show()

# Insight: does median price consistently rise with bedroom count?
print('Median price by bedrooms:')
print(df.groupby('bedrooms')['price'].median().apply(lambda x: f'${x:,.0f}'))

### 2.3 `sns.violinplot` — Full Distribution Shape per Neighbourhood

A **violin plot** is like a box plot but also shows the full distribution shape (KDE). It is better than a box plot when:
- You suspect bimodal distributions (two peaks)
- Sample sizes are large enough to estimate a smooth density
- The shape of the distribution matters, not just the quartiles

In [ ]:
# ── sns.violinplot: price distribution shape per neighbourhood ────────────────
fig, ax = plt.subplots(figsize=(12, 6))

sns.violinplot(
    data=df,
    x='neighborhood',
    y='price',
    palette='husl',
    inner='quartile',   # show quartile lines inside the violin
    cut=0,              # do not extend the violin beyond data range
    ax=ax
)

# Overlay the actual median price as a text annotation
for i, hood in enumerate(df['neighborhood'].unique()):
    median_val = df[df['neighborhood'] == hood]['price'].median()
    ax.text(i, median_val + 20_000, f'${median_val/1000:.0f}K',
            ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_title('Price Distribution Shape by Neighbourhood (Violin Plot)', fontweight='bold')
ax.set_xlabel('Neighbourhood')
ax.set_ylabel('Sale Price ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

# Explanation of reading a violin:
# Width ∝ density of houses at that price level
# Wider at $300K = more houses sold around that price
# Long tail upward = a few very expensive houses
plt.tight_layout()
plt.show()

### 2.4 `sns.heatmap` — Correlation Matrix

**When to use:** Show the pairwise correlation between all numeric features.

**How to read it:**
- Each cell (i, j) shows the Pearson correlation between feature i and feature j
- **+1.0** = perfect positive correlation (dark red/warm colour)
- **0.0** = no linear relationship (white/neutral)
- **–1.0** = perfect negative correlation (dark blue/cool colour)

**ML significance:**
- Features highly correlated with the target (price) are likely to be good predictors
- Features highly correlated with each other → multicollinearity → may need to drop one

In [ ]:
# ── Correlation heatmap ───────────────────────────────────────────────────────
# Select only numeric columns (drop house_id string and non-numeric)
numeric_cols = ['sqft', 'bedrooms', 'bathrooms', 'age_years', 'garage',
                'price', 'price_per_sqft', 'log_price']
corr_matrix = df[numeric_cols].corr()  # Pearson correlation by default

fig, ax = plt.subplots(figsize=(10, 8))

sns.heatmap(
    corr_matrix,
    annot=True,           # print the correlation value in each cell
    fmt='.2f',            # format numbers to 2 decimal places
    cmap='coolwarm',      # blue (negative) → white (zero) → red (positive)
    center=0,             # centre colour scale at 0
    vmin=-1, vmax=1,      # fix scale to [-1, 1] range
    square=True,          # make each cell square
    linewidths=0.5,       # thin lines between cells
    linecolor='white',
    cbar_kws={'label': 'Pearson r', 'shrink': 0.8},
    ax=ax
)

ax.set_title('Feature Correlation Matrix — House Price Dataset',
             fontweight='bold', pad=15)

# Rotate axis labels for readability
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)

plt.tight_layout()
plt.show()

# Print top correlations with price
print('\nTop 5 features correlated with price:')
print(corr_matrix['price'].abs().sort_values(ascending=False).head(6))

### 2.5 `sns.pairplot` — Scatter Matrix for All Feature Pairs

A **pair plot** creates a grid where:
- Off-diagonal cells = scatter plots for each pair of features
- Diagonal cells = distribution (histogram or KDE) of each single feature

With N features, you get N × N cells — showing all N(N-1)/2 unique pairwise relationships at once. This is your fastest EDA tool for datasets with fewer than ~10 features.

In [ ]:
# ── Pairplot: 5 key features coloured by neighbourhood ────────────────────────
# Limit to 5 features — with more features the grid becomes too small to read
pair_cols = ['sqft', 'bedrooms', 'age_years', 'price', 'price_per_sqft']

pair_grid = sns.pairplot(
    df[pair_cols + ['neighborhood']],
    hue='neighborhood',          # colour points by neighbourhood
    diag_kind='kde',             # KDE on diagonal instead of histogram
    plot_kws={'alpha': 0.5, 's': 20},    # point transparency and size
    diag_kws={'fill': True, 'alpha': 0.4},
    palette='husl'
)

pair_grid.figure.suptitle(
    'Pairwise Relationships — House Price Features',
    y=1.01, fontsize=13, fontweight='bold'
)

plt.tight_layout()
plt.show()

# What to look for:
# sqft vs price: should show a clear positive diagonal band
# age_years vs price: should show a weak negative trend
# bedrooms vs price: step-wise positive relationship

### 2.6 `sns.regplot` — Scatter with Regression Line

`regplot` combines a scatter plot with a **fitted regression line** and a **95 % confidence interval band**. This is the first visual check before fitting a linear regression model.

In [ ]:
# ── sns.regplot: sqft vs price with regression line ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: linear regression on raw price
sns.regplot(
    data=df,
    x='sqft',
    y='price',
    scatter_kws={'alpha': 0.4, 's': 30, 'color': 'steelblue'},   # style the dots
    line_kws={'color': 'tomato', 'lw': 2},                       # style the line
    ci=95,           # shade 95% confidence interval around the regression line
    ax=axes[0]
)
axes[0].set_title('sqft vs Price (Linear Regression)', fontweight='bold')
axes[0].set_xlabel('House Size (sq ft)')
axes[0].set_ylabel('Sale Price ($)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

# Right: same but using log_price on y-axis
# If the residuals around the log-price line look tighter → log is a better target
sns.regplot(
    data=df,
    x='sqft',
    y='log_price',
    scatter_kws={'alpha': 0.4, 's': 30, 'color': 'mediumseagreen'},
    line_kws={'color': 'darkorange', 'lw': 2},
    ci=95,
    ax=axes[1]
)
axes[1].set_title('sqft vs log(Price) (Linear Regression)', fontweight='bold')
axes[1].set_xlabel('House Size (sq ft)')
axes[1].set_ylabel('log(Sale Price)')

plt.suptitle('regplot: Checking Linearity Before Modelling',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 2.7 `sns.barplot` — Mean with Confidence Intervals

Unlike a plain bar chart, `sns.barplot` automatically computes the **mean** and draws **95 % confidence interval error bars**. This shows not just the average but also how certain you are about it.

In [ ]:
# ── sns.barplot: mean price by neighbourhood with CI error bars ────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: mean price per neighbourhood
sns.barplot(
    data=df,
    x='neighborhood',
    y='price',
    palette='husl',
    errorbar='ci',      # 'ci' = 95% confidence interval
    capsize=0.1,        # small horizontal cap on the error bar
    ax=axes[0]
)
axes[0].set_title('Mean Price by Neighbourhood\n(with 95% CI)', fontweight='bold')
axes[0].set_xlabel('Neighbourhood')
axes[0].set_ylabel('Mean Sale Price ($)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

# Right: mean price_per_sqft by bedrooms
sns.barplot(
    data=df,
    x='bedrooms',
    y='price_per_sqft',
    palette='husl',
    errorbar='ci',
    capsize=0.1,
    ax=axes[1]
)
axes[1].set_title('Mean Price/sqft by Bedrooms\n(with 95% CI)', fontweight='bold')
axes[1].set_xlabel('Number of Bedrooms')
axes[1].set_ylabel('Mean Price per sq ft ($)')

# Interpretation: narrow CI = more confident about the mean
# (more data in that group → tighter CI)
plt.tight_layout()
plt.show()

## Part 3 — Chart Selection Guide

### When to use each chart type

| Question | Data type | Best chart |
|----------|-----------|------------|
| How is one variable distributed? | 1 continuous | Histogram, KDE, Box plot |
| How do two distributions compare? | 1 continuous × 1 categorical | Box plot, Violin plot |
| Is there a relationship between X and Y? | 2 continuous | Scatter plot, Regplot |
| How do categories compare on one measure? | 1 categorical × 1 continuous | Bar chart |
| How does a value change over time? | time × continuous | Line plot |
| How are ALL features related to each other? | N continuous | Pairplot, Heatmap |
| Is there a trend + uncertainty? | 2 continuous + CI | Regplot, Line plot with CI |
| 3 variables at once? | 2 continuous + 1 categorical | Scatter with hue |

### Continuous vs Categorical decision tree
```
How many variables?
│
├─ ONE variable
│  ├─ Continuous → Histogram / KDE
│  └─ Categorical → Bar chart (counts)
│
├─ TWO variables
│  ├─ Both continuous → Scatter / Regplot
│  ├─ Cont × Cat → Box / Violin / Bar
│  └─ Both categorical → Grouped bar / Heatmap
│
└─ MANY variables → Pairplot / Correlation Heatmap
```

## Part 4 — Combined EDA Dashboard (2 × 3 Grid)

This is what a professional EDA report looks like: a single figure with 6 complementary charts that together tell the full story of the dataset.

In [ ]:
# ── 6-panel EDA dashboard ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('House Price Dataset — Complete EDA Dashboard',
             fontsize=16, fontweight='bold', y=1.01)

# ── Panel 1 (row 0, col 0): Price histogram with KDE ─────────────────────────
sns.histplot(data=df, x='price', kde=True, bins=25,
             color='steelblue', ax=axes[0, 0])
axes[0, 0].axvline(df['price'].median(), color='tomato', lw=2,
                   linestyle='--', label=f'Median ${df["price"].median()/1000:.0f}K')
axes[0, 0].set_title('1. Price Distribution', fontweight='bold')
axes[0, 0].set_xlabel('Sale Price ($)')
axes[0, 0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
axes[0, 0].legend(fontsize=9)

# ── Panel 2 (row 0, col 1): Scatter sqft vs price ────────────────────────────
sns.scatterplot(data=df, x='sqft', y='price', hue='neighborhood',
                alpha=0.55, s=35, palette='husl', ax=axes[0, 1])
# Annotate the most expensive house
top = df.loc[df['price'].idxmax()]
axes[0, 1].annotate(f"{top['house_id']}\n${top['price']/1000:.0f}K",
                     xy=(top['sqft'], top['price']),
                     xytext=(top['sqft'] - 700, top['price'] - 80_000),
                     arrowprops=dict(arrowstyle='->', color='black'),
                     fontsize=8)
axes[0, 1].set_title('2. Size vs Price (by Neighbourhood)', fontweight='bold')
axes[0, 1].set_xlabel('Size (sq ft)')
axes[0, 1].set_ylabel('Price ($)')
axes[0, 1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
axes[0, 1].legend(title='Hood', fontsize=7, title_fontsize=8)

# ── Panel 3 (row 0, col 2): Box plot price by bedrooms ───────────────────────
sns.boxplot(data=df, x='bedrooms', y='price', palette='husl',
            width=0.5, ax=axes[0, 2])
axes[0, 2].set_title('3. Price by Bedrooms', fontweight='bold')
axes[0, 2].set_xlabel('Bedrooms')
axes[0, 2].set_ylabel('Price ($)')
axes[0, 2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

# ── Panel 4 (row 1, col 0): Violin plot by neighbourhood ─────────────────────
sns.violinplot(data=df, x='neighborhood', y='price', palette='husl',
               inner='quartile', cut=0, ax=axes[1, 0])
axes[1, 0].set_title('4. Price Shape by Neighbourhood', fontweight='bold')
axes[1, 0].set_xlabel('Neighbourhood')
axes[1, 0].set_ylabel('Price ($)')
axes[1, 0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
axes[1, 0].tick_params(axis='x', labelrotation=10)

# ── Panel 5 (row 1, col 1): Correlation heatmap ──────────────────────────────
mini_corr = df[['sqft', 'bedrooms', 'age_years', 'garage', 'price']].corr()
sns.heatmap(mini_corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, cbar=False,
            ax=axes[1, 1])
axes[1, 1].set_title('5. Feature Correlations', fontweight='bold')
axes[1, 1].tick_params(axis='x', labelrotation=30)

# ── Panel 6 (row 1, col 2): Regplot sqft vs log price ────────────────────────
sns.regplot(data=df, x='sqft', y='log_price',
            scatter_kws={'alpha': 0.4, 's': 25, 'color': 'mediumseagreen'},
            line_kws={'color': 'darkorange', 'lw': 2},
            ci=95, ax=axes[1, 2])
axes[1, 2].set_title('6. Size vs log(Price) + Regression', fontweight='bold')
axes[1, 2].set_xlabel('Size (sq ft)')
axes[1, 2].set_ylabel('log(Price)')

plt.tight_layout(rect=[0, 0, 1, 0.99])
plt.show()

## 5. Summary — Matplotlib vs Seaborn Cheat Sheet

```python
# ── matplotlib (full control) ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x, y, color='steelblue', linewidth=2, marker='o')
ax.scatter(x, y, alpha=0.5, s=40)
ax.hist(data, bins=30, edgecolor='white')
ax.bar(categories, values)
ax.set_title('Title', fontweight='bold')
ax.set_xlabel('X Label'); ax.set_ylabel('Y Label')
ax.legend()
plt.tight_layout(); plt.show()

# ── seaborn (statistical, concise) ────────────────────────────────────
sns.set_theme(style='whitegrid', palette='husl')
fig, ax = plt.subplots(figsize=(10, 5))

sns.histplot(data=df, x='price', kde=True, hue='neighborhood', ax=ax)
sns.boxplot(data=df, x='bedrooms', y='price', palette='husl', ax=ax)
sns.violinplot(data=df, x='neighborhood', y='price', inner='quartile', ax=ax)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
sns.scatterplot(data=df, x='sqft', y='price', hue='neighborhood', ax=ax)
sns.regplot(data=df, x='sqft', y='price', ci=95, ax=ax)
sns.barplot(data=df, x='neighborhood', y='price', errorbar='ci', ax=ax)
sns.pairplot(df[['sqft','bedrooms','price']], hue='neighborhood')
```

## 6. Self-Check Questions

---

### Q1. When would you prefer a violin plot over a box plot?

<details>
<summary>Click to reveal answer</summary>

Choose a **violin plot** over a box plot when:

1. **Distribution shape matters** — you suspect the data is bimodal (two peaks) or has a non-standard shape. A box plot only shows quartiles; a violin shows the full KDE.
2. **Large sample sizes** — KDE is unreliable with < 30 points. With 50+ per group, the violin shape is meaningful.
3. **Comparing spread** — you want to see whether one group has a wide, flat distribution (uniform prices) vs a narrow, tall one (concentrated prices).

Stick with **box plots** when sample sizes are small (KDE would be misleading) or when you specifically need to highlight outliers (box plots flag them explicitly as individual points).
</details>

---

### Q2. You want to show the relationship between 5 features simultaneously. What chart shows all pairwise relationships at once?

<details>
<summary>Click to reveal answer</summary>

A **pairplot** (scatter matrix) — `sns.pairplot()` in seaborn.

With 5 features it creates a 5 × 5 grid:
- **Diagonal** (5 cells): distribution of each individual feature (histogram or KDE)
- **Off-diagonal** (20 cells): scatter plot for every pair of features

Total unique pairwise relationships shown: 5 × (5–1) / 2 = **10 scatter plots**.

Adding `hue='neighborhood'` colours points by category, revealing whether relationships differ across groups.
</details>

---

### Q3. In a correlation heatmap, cell (i, j) shows 0.85. What does this mean, and what colour would it likely be?

<details>
<summary>Click to reveal answer</summary>

**Meaning:** Feature i and feature j have a **strong positive linear correlation** of 0.85. As one increases, the other tends to increase proportionally. For example, `sqft` and `price` might show ~0.85 — larger houses generally cost more.

**Colour:** With `cmap='coolwarm'` and `center=0`, positive values appear **red/warm**. A value of 0.85 (close to the maximum of 1.0) would appear as a **deep red**.

**ML implication:** If two *features* (not feature and target) have correlation 0.85, they are carrying nearly the same information. You might consider dropping one to reduce multicollinearity before fitting a linear model.
</details>

---

### Q4. Your histogram shows price distribution is right-skewed. What can you do to make it more symmetric for visualisation?

<details>
<summary>Click to reveal answer</summary>

Apply a **log transform**: `df['log_price'] = np.log(df['price'])`

**Why this works:** Right-skewed data means a small number of very large values stretch the tail to the right. The logarithm compresses large values (e.g., log(1 000 000) = 13.8, log(100 000) = 11.5) while spreading out small values — this "pulls in" the right tail and creates a more symmetric distribution.

Other options:
- **Square root transform** (`np.sqrt`) — gentler than log, good for moderate skew
- **Box-Cox transform** (`scipy.stats.boxcox`) — finds the optimal power transform automatically
- **Clip outliers** — cap at the 99th percentile before plotting (but do NOT clip for modelling)

**Important:** For visualisation, transform freely. For ML, be consistent — if you log-transform price for training, you must invert the transform (use `np.exp`) on model predictions.
</details>

In [ ]:
# ── Bonus: quick reference — all 8 chart types in one figure ──────────────────
# This cell serves as a visual cheat sheet for the notebook

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle('Matplotlib & Seaborn — Chart Type Reference', fontsize=15, fontweight='bold')

# 1. Line plot
axes[0, 0].plot(monthly_avg['month'], monthly_avg['avg_price'],
                marker='o', color='steelblue', lw=2)
axes[0, 0].set_title('Line Plot\n(trend over time)')
axes[0, 0].set_xlabel('Month'); axes[0, 0].set_ylabel('Avg Price')
axes[0, 0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

# 2. Scatter plot
axes[0, 1].scatter(df['sqft'], df['price'], alpha=0.3, s=15, color='steelblue')
axes[0, 1].set_title('Scatter Plot\n(two continuous vars)')
axes[0, 1].set_xlabel('sqft'); axes[0, 1].set_ylabel('Price')
axes[0, 1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

# 3. Histogram
axes[0, 2].hist(df['price'], bins=25, color='mediumseagreen', edgecolor='white')
axes[0, 2].set_title('Histogram\n(one continuous var)')
axes[0, 2].set_xlabel('Price')
axes[0, 2].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

# 4. Bar chart
avg_p = df.groupby('neighborhood')['price'].mean()
axes[0, 3].bar(avg_p.index, avg_p.values, color=sns.color_palette('husl', 4), edgecolor='black', lw=0.5)
axes[0, 3].set_title('Bar Chart\n(categorical comparison)')
axes[0, 3].set_ylabel('Avg Price')
axes[0, 3].tick_params(axis='x', rotation=20)
axes[0, 3].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

# 5. Box plot
sns.boxplot(data=df, x='bedrooms', y='price', palette='husl', width=0.5, ax=axes[1, 0])
axes[1, 0].set_title('Box Plot\n(distribution by category)')
axes[1, 0].set_xlabel('Bedrooms'); axes[1, 0].set_ylabel('Price')
axes[1, 0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

# 6. Violin plot
sns.violinplot(data=df, x='neighborhood', y='price', palette='husl',
               inner='quartile', cut=0, ax=axes[1, 1])
axes[1, 1].set_title('Violin Plot\n(distribution shape)')
axes[1, 1].set_xlabel(''); axes[1, 1].set_ylabel('Price')
axes[1, 1].tick_params(axis='x', rotation=20)
axes[1, 1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

# 7. Heatmap
mini_c = df[['sqft','bedrooms','age_years','price']].corr()
sns.heatmap(mini_c, annot=True, fmt='.1f', cmap='coolwarm',
            center=0, square=True, cbar=False, ax=axes[1, 2])
axes[1, 2].set_title('Heatmap\n(correlation matrix)')

# 8. Regression plot
sns.regplot(data=df, x='sqft', y='price',
            scatter_kws={'alpha': 0.3, 's': 15, 'color': 'steelblue'},
            line_kws={'color': 'tomato', 'lw': 2},
            ci=95, ax=axes[1, 3])
axes[1, 3].set_title('Regplot\n(scatter + regression line)')
axes[1, 3].set_xlabel('sqft'); axes[1, 3].set_ylabel('Price')
axes[1, 3].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()